# Ch.14 — Unsupervised Metrics

> **Running theme:** The real estate platform ran K-Means, DBSCAN, and HDBSCAN in Ch.12 — but which clustering was actually good? Without labelled districts, there is no accuracy or F1. Unsupervised metrics measure geometric properties of the clusters themselves. This chapter introduces the three internal metrics and the external Adjusted Rand Index.

## Core Idea

**Internal metrics** (no labels needed):
- **Silhouette score:** $(b - a) / \max(a, b)$ — range $[-1, 1]$, higher is better. $a$ = mean intra-cluster distance, $b$ = mean distance to nearest cluster.
- **Davies-Bouldin index (DBI):** mean of best (intra scatter / inter centroid distance) ratios — lower is better.
- **Calinski-Harabasz index (CHI):** between-cluster / within-cluster dispersion ratio — higher is better.

**External metric** (requires a reference labelling):
- **Adjusted Rand Index (ARI):** corrected-for-chance concordance with ground truth — range $[-1, 1]$, higher is better.

## Running Example

Dataset: **California Housing** (sklearn) — 20,640 census districts, 8 features (standardised).  
Clustering: K-Means (K=2…15, K-Means++ init).  
External proxy: latitude × longitude quantile bins — 16 rough geographic quadrants.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)

# ── Load and scale ─────────────────────────────────────────────────────────────
housing = fetch_california_housing()
X       = housing.data
scaler  = StandardScaler()
X_sc    = scaler.fit_transform(X)

print(f"Dataset: {X.shape[0]:,} samples × {X.shape[1]} features")

## K Sweep: Computing All Three Internal Metrics

Silhouette is $O(n^2)$ — use `sample_size=5000`. DBI and CHI are fast.

In [ ]:
K_range    = range(2, 16)
sil_list   = []
dbi_list   = []
chi_list   = []
km_labels  = {}   # cache labels for later use

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_sc)
    km_labels[k] = km.labels_

    sil_list.append(silhouette_score(X_sc, km.labels_, sample_size=5000, random_state=42))
    dbi_list.append(davies_bouldin_score(X_sc, km.labels_))
    chi_list.append(calinski_harabasz_score(X_sc, km.labels_))

Ks = list(K_range)
print(f"Best K by silhouette         : {Ks[np.argmax(sil_list)]}  ({max(sil_list):.4f})")
print(f"Best K by Davies-Bouldin     : {Ks[np.argmin(dbi_list)]}  ({min(dbi_list):.4f})")
print(f"Best K by Calinski-Harabasz  : {Ks[np.argmax(chi_list)]}  ({max(chi_list):.0f})")

## Three-Metric Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (axes[0], sil_list, 'Silhouette score (↑ better)', 'b', 'argmax'),
    (axes[1], dbi_list, 'Davies-Bouldin index (↓ better)', 'r', 'argmin'),
    (axes[2], chi_list, 'Calinski-Harabász index (↑ better)', 'g', 'argmax'),
]

for ax, values, title, colour, best_fn in configs:
    best_idx = np.argmin(values) if best_fn == 'argmin' else np.argmax(values)
    ax.plot(Ks, values, f'{colour}-o', markersize=5)
    ax.axvline(Ks[best_idx], color='orange', linestyle='--', lw=1.5,
               label=f'best K={Ks[best_idx]}')
    ax.set_xlabel('K'); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Three internal metrics across K — look for consensus', y=1.02)
plt.tight_layout(); plt.show()

## Silhouette Subplot (Per-Cluster Bars)

The mean silhouette score hides internal variation. Plot per-cluster silhouette bars to spot clusters with negative values — those contain misassigned points and the K should be reconsidered.

In [ ]:
from sklearn.metrics import silhouette_samples

Ks_to_plot = [3, 5, 7]
fig, axes  = plt.subplots(1, len(Ks_to_plot), figsize=(15, 5))

for ax, k in zip(axes, Ks_to_plot):
    labels    = km_labels[k]
    sil_samp  = silhouette_samples(X_sc, labels)
    mean_sil  = sil_samp.mean()

    y_lower = 10
    for cluster_id in range(k):
        cluster_sil = np.sort(sil_samp[labels == cluster_id])
        y_upper = y_lower + len(cluster_sil)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                          alpha=0.7)
        y_lower = y_upper + 10

    ax.axvline(mean_sil, color='red', linestyle='--', lw=1.5)
    ax.set_xlim([-0.2, 0.6])
    ax.set_title(f'K={k}  mean s={mean_sil:.3f}')
    ax.set_xlabel('Silhouette coefficient')
    ax.set_yticks([])
    ax.grid(True, alpha=0.3)

plt.suptitle('Silhouette subplot — clusters with negative bars are poorly separated', y=1.02)
plt.tight_layout(); plt.show()

## External Validation: ARI and NMI vs Geographic Proxy

We don't have true labels. But latitude and longitude encode geography — a reasonable proxy for natural district groupings. We create 16 geo-quadrants by binning lat/lon at tertiles.

In [ ]:
# ── Proxy labels: 4-class lat × 4-class lon → 16 geo-quadrants ───────────────
lat = X[:, 6]   # Latitude
lon = X[:, 7]   # Longitude

lat_bin = np.digitize(lat, np.percentile(lat, [25, 50, 75]))   # 0, 1, 2, 3
lon_bin = np.digitize(lon, np.percentile(lon, [25, 50, 75]))   # 0, 1, 2, 3
proxy   = lat_bin * 4 + lon_bin                                # 16 classes

print(f"Proxy label distribution: {np.unique(proxy, return_counts=True)[1]}")

# ── ARI and NMI for each K ────────────────────────────────────────────────────
ari_list = []
nmi_list = []
for k in Ks:
    ari_list.append(adjusted_rand_score(proxy, km_labels[k]))
    nmi_list.append(normalized_mutual_info_score(proxy, km_labels[k]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(Ks, ari_list, 'b-o', markersize=5)
ax1.axhline(0, color='gray', linestyle=':', lw=1)
ax1.set_title('ARI vs geo-proxy labels')
ax1.set_xlabel('K'); ax1.set_ylabel('Adjusted Rand Index'); ax1.grid(True, alpha=0.3)

ax2.plot(Ks, nmi_list, 'r-o', markersize=5)
ax2.set_title('NMI vs geo-proxy labels')
ax2.set_xlabel('K'); ax2.set_ylabel('Normalised Mutual Information'); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Best ARI at K={Ks[np.argmax(ari_list)]}  ({max(ari_list):.4f})")
print(f"Best NMI at K={Ks[np.argmax(nmi_list)]}  ({max(nmi_list):.4f})")

## Hyperparameter Dial: Silhouette `sample_size` Impact on Runtime

In [ ]:
import time

km5     = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X_sc)
true_sil = silhouette_score(X_sc, km5.labels_)   # full computation
print(f"Full silhouette (n={len(X_sc):,}): {true_sil:.6f}")

print("\nSampled estimates vs full:")
for ss in [500, 1000, 2000, 5000, 10000]:
    t0  = time.perf_counter()
    s   = silhouette_score(X_sc, km5.labels_, sample_size=ss, random_state=42)
    dt  = time.perf_counter() - t0
    print(f"  sample_size={ss:>6}: sil={s:.6f}  Δ={abs(s-true_sil):.6f}  time={dt:.3f}s")

## Metric Consensus: Which K Do They All Agree On?

In [ ]:
# Normalise all metrics to [0, 1] so they can be compared on the same axis
# For DBI: invert (lower = better → 1 - normalised)
def minmax(arr): return (np.array(arr) - min(arr)) / (max(arr) - min(arr))

norm_sil = minmax(sil_list)
norm_dbi = 1 - minmax(dbi_list)   # inverted: lower DBI is better
norm_chi = minmax(chi_list)
consensus = norm_sil + norm_dbi + norm_chi

best_consensus_k = Ks[np.argmax(consensus)]

plt.figure(figsize=(10, 4))
plt.plot(Ks, norm_sil, 'b-o', label='Silhouette (norm)', markersize=5)
plt.plot(Ks, norm_dbi, 'r-o', label='DBI (inverted norm)', markersize=5)
plt.plot(Ks, norm_chi, 'g-o', label='CHI (norm)', markersize=5)
plt.plot(Ks, consensus / 3, 'k--o', label='Consensus (mean)', markersize=5, lw=2)
plt.axvline(best_consensus_k, color='orange', linestyle=':', lw=2,
            label=f'Best consensus K={best_consensus_k}')
plt.xlabel('K'); plt.ylabel('Normalised score'); plt.legend()
plt.title('Normalised metric consensus across K')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f"Consensus selects K = {best_consensus_k}")

## What Can Go Wrong: DBI Applied to DBSCAN

DBI requires centroids. DBSCAN doesn't produce centroids. Applying it anyway produces a misleading score.

In [ ]:
from sklearn.cluster import DBSCAN

db     = DBSCAN(eps=0.5, min_samples=16, algorithm='ball_tree', n_jobs=-1).fit(X_sc)
labels = db.labels_
n_c    = len(set(labels)) - (1 if -1 in labels else 0)

# Filter out noise points for the metrics (noise label = -1)
mask        = labels != -1
labels_clean = labels[mask]
X_clean      = X_sc[mask]

if len(set(labels_clean)) >= 2:
    dbi_db = davies_bouldin_score(X_clean, labels_clean)
    sil_db = silhouette_score(X_clean, labels_clean, sample_size=5000, random_state=42)
    print(f"DBSCAN: {n_c} clusters, {(labels==-1).sum():,} noise points filtered out")
    print(f"  Silhouette (non-noise only): {sil_db:.4f}")
    print(f"  DBI (non-noise only):        {dbi_db:.4f}")
    print("\n⚠ DBI uses cluster means as centroids — valid approximation but not interpretable")
    print("  for density-based clusters. Compare with caution.")
else:
    print("Too few non-noise clusters to compute metrics — try adjusting eps")

## Exercises

1. **Metrics on HDBSCAN.** Fit HDBSCAN (if installed) with `min_cluster_size=200`. Compute silhouette score (excluding noise points). Compare to K-Means at the consensus K. Which clusters score higher?

2. **Consensus K vs domain K.** The business team insists on K=3 (low, mid, high value). Compute DBI and silhouette at K=3 and at the consensus K. How much metric quality is lost by the domain constraint? Is the loss worth the interpretability gain?

3. **ARI with noise.** Re-run the ARI comparison but this time randomly shuffle 20% of the K-Means labels before computing ARI (simulating label noise). Plot ARI before and after shuffling for K=2 to 15. How sensitive is ARI to noise in the predicted labels?

In [ ]:
# Exercise 1 — Metrics on HDBSCAN
# TODO: your solution here

In [ ]:
# Exercise 2 — Consensus K vs domain K
# TODO: your solution here

In [ ]:
# Exercise 3 — ARI sensitivity to label noise
# TODO: your solution here